In [2]:
import numpy as np
import torch
from sklearn.datasets import load_sample_images

sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255
sample_images.shape

torch.Size([2, 427, 640, 3])

In [3]:
sample_images_permuted = sample_images.permute(0,3,1,2)
sample_images_permuted.shape

torch.Size([2, 3, 427, 640])

In [4]:
import torchvision
import torchvision.transforms.v2 as T
cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

In [5]:
import torch.nn as nn

torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7, padding="same")
fmaps = conv_layer(cropped_images)

In [6]:
fmaps.shape

torch.Size([2, 32, 70, 120])

In [7]:
conv_layer.weight.shape

torch.Size([32, 3, 7, 7])

In [8]:
conv_layer.bias.shape

torch.Size([32])

In [9]:
max_pool = nn.MaxPool2d(kernel_size=2)

In [10]:
from typing import Any

import torch.nn.functional as F

class DepthPool(torch.nn.Module):
    def __init__(self, kernel_size:int, stride:int = -1, padding = 0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride != -1 else kernel_size
        self.padding = padding
    
    def forward(self, inputs):
        batch, channels, height, width = inputs.shape
        Z = inputs.view(batch, channels, height * width)
        Z = Z.permute(0,2,1)
        Z = F.max_pool1d(Z, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding)
        Z = Z.permute(0,2,1)
        return Z.view(batch, -1, height, width)

In [11]:
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)

In [13]:
from functools import partial

device = "cuda"
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")
model = nn.Sequential(
    DefaultConv2d(1, 64, kernel_size=7), nn.ReLU(),
    nn.MaxPool2d(2),
    DefaultConv2d(64, 128), nn.ReLU(),
    DefaultConv2d(128, 128), nn.ReLU(),
    nn.MaxPool2d(2),
    DefaultConv2d(128, 256), nn.ReLU(),
    DefaultConv2d(256, 256), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(2304, 128), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(64, 10)
).to(device)